# Graph Summarization Algorithm

This notebook demonstrates a simplified DAG-based graph summarization algorithm applied to a CFG tha should be effectively the same as summarize_graph.py.

## Algorithm Overview

1. Initialize empty summaries for all functions/communities
2. Use GreedyFAS to remove the minimum number of edges to create an acyclic graph (DAG) (using `nx.minimum_feedback_arc_set`), which is required to compute a topological sort.
3. Compute reverse topological sort to determine processing order (using `nx.topological_sort`) 
4. Process functions sequentially. The topological sort step (3) guarantees that dependencies are summarized first, so they can be used as context for subsequent summarizations.


In [1]:
import networkx as nx
import matplotlib.pyplot as plt
from ipysigma import Sigma
import numpy as np
from collections import defaultdict

## 0. Load CFG

In [4]:
from src.visualize_cfg import load_cfg

G = load_cfg('reorder_and_pad2.json')

## 1. Detect and Aggregate Communities

For now I'm just using Leiden on the individual block nodes, creating more even sized units to be summarized, although they may not correspond to logical units as well as just using functions. We can investigate later if its better to start with entire functions or not.

In [33]:
from src.community_detection import collapse_leiden
G_communities, community_list = collapse_leiden(G, resolution=0.001)

widget = Sigma(
    G_communities,
    node_label='label',
    raw_node_color=lambda n, d: "red" if "main" in d.get("func") else d.get('color'),
    edge_color='type',
    label_font='sans-serif',
    default_edge_type='arrow',
    start_layout=10,
)
display(widget)

Sigma(nx.DiGraph with 171 nodes and 478 edges)

## 2. Remove Cycles using GreedyFAS

The minimum feedback arc set problem (FAS) is NP-hard so tihs is a vibe-coded implementation of GreedyFAS ([paper](https://arxiv.org/pdf/2208.09234)), an approximate version. It reomves the minimum number of edges in a directed graph to make it acyclic.



In [34]:
import networkx as nx

def greedy_feedback_arc_set(G):
    feedback_edges = set()
    visited = set()
    rec_stack = set()
    
    def dfs(node):
        visited.add(node)
        rec_stack.add(node)
        
        for neighbor in G.successors(node):
            if neighbor not in visited:
                dfs(neighbor)
            elif neighbor in rec_stack:
                # Back edge found - add to feedback arc set
                feedback_edges.add((node, neighbor))
        
        rec_stack.remove(node)
    
    for node in G.nodes():
        if node not in visited:
            dfs(node)
    
    return feedback_edges

# Apply greedy FAS
fas_edges = greedy_feedback_arc_set(G_communities)

# Create DAG by removing FAS edges
G_dag = G_communities.copy()
G_dag.remove_edges_from(fas_edges)

widget = Sigma(
    G_dag,
    node_label='label',
    raw_node_color=lambda n, d: "red" if "main" in d.get("func") else d.get('color'),
    edge_color='type',
    label_font='sans-serif',
    default_edge_type='arrow',
    start_layout=10,
)
display(widget)

print(f"FAS size: {len(fas_edges)} edges")
print(f"Before FAS removal: {G_communities.number_of_nodes()} nodes, {G_communities.number_of_edges()} edges")
print(f"After FAS removal: {G_dag.number_of_nodes()} nodes, {G_dag.number_of_edges()} edges")
    

Sigma(nx.DiGraph with 171 nodes and 316 edges)

FAS size: 162 edges
Before FAS removal: 171 nodes, 478 edges
After FAS removal: 171 nodes, 316 edges


## 3. Compute Reverse Topological Sort

Now that the graph is a DAG, we can find a topological sort, and simply summarize them in reverse order. This guarantees that every node's dependencies/children are summarized before the node itself, without any tricky graph traversal stuff.

In [21]:
summary_order = list(reversed(list(nx.topological_sort(G_dag))))
print(summary_order)
print(len(summary_order))

[30, 67, 16, 85, 10, 95, 83, 62, 47, 50, 139, 146, 157, 123, 107, 72, 126, 15, 39, 142, 133, 140, 55, 32, 113, 89, 93, 68, 42, 144, 26, 131, 69, 59, 77, 31, 128, 58, 75, 122, 81, 12, 154, 117, 100, 19, 9, 24, 88, 28, 53, 101, 25, 105, 35, 52, 120, 111, 63, 41, 4, 23, 11, 18, 54, 104, 40, 60, 14, 6, 74, 51, 116, 91, 17, 3, 115, 86, 48, 46, 5, 90, 78, 2, 34, 1, 112, 136, 64, 57, 37, 70, 13, 84, 82, 22, 110, 143, 73, 61, 20, 44, 141, 38, 79, 158, 159, 43, 29, 87, 138, 7, 99, 134, 36, 97, 119, 109, 45, 66, 114, 21, 65, 76, 27, 49, 102, 71, 106, 56, 8, 174, 173, 172, 171, 170, 169, 168, 167, 166, 165, 164, 163, 162, 161, 160, 156, 155, 153, 152, 151, 150, 149, 148, 147, 145, 137, 135, 132, 130, 129, 127, 125, 124, 121, 118, 108, 103, 98, 96, 94, 92, 80, 33, 0]
175


## 4. Summarize with LLM in Reverse Topological Order

In [ ]:
import ollama
from tqdm import tqdm
import json

# Configure LLM
model = "qwen2.5-coder:7b"
base_url = "http://100.104.79.110:11434"
client = ollama.Client(host=base_url)

# Process all nodes
for node in tqdm(summary_order):
    predecessors = list(G_dag.predecessors(node))
    dependency_summaries = [(pred, G_communities.nodes[pred].get('summary', "")) for pred in predecessors]
    
    prompt = f"Summarize the function/node '{node}' based on its dependencies:\n"
    for pred, summary in dependency_summaries:
        if summary:
            prompt += f"- {pred}: {summary}\n"
    prompt += f"\nGenerate a concise summary for {node}:"
    
    response = client.generate(model=model, prompt=prompt, stream=False)
    summary = response["response"]
    G_communities.nodes[node]['summary'] = summary

# Save summaries
summaries = {node: G_communities.nodes[node]['summary'] for node in G_communities.nodes() if 'summary' in G_communities.nodes[node]}
with open('summaries.json', 'w') as f:
    json.dump(summaries, f, indent=2)
print(f"Summarization complete. Saved {len(summaries)} summaries to summaries.json")

  0%|          | 0/175 [00:00<?, ?it/s]

100%|██████████| 175/175 [01:36<00:00,  1.81it/s]

Summarization complete. Checkpoint saved.
